MAD OFF

In [36]:
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    pairwise_distances
)
from sklearn.cluster import KMeans
import openpyxl

# --------------------------
# ENV / DETERMINISM SETTINGS
# --------------------------
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

warnings.filterwarnings("ignore")
MASTER_SEED = 42
random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
GLOBAL_RNG = np.random.default_rng(MASTER_SEED)

# --------------------------
# USER CONFIG - EDIT THIS
# --------------------------
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/circles.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/fashion-mnist_test.csv"  # <-- replace with your dataset path  # <-- replace with your dataset path
#
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/file_1.csv"  # <-- replace with your dataset path
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/cluster_data.csv"  # <-- replace with your dataset path
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/Dry_Bean_Dataset.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/heart_failure_clinical_records_dataset.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/AirfoilSelfNoise.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/Rice_data_type.csv"  # <-- replace with your dataset path

#-----------------------------------------------------------------------------------------------------------------------------------------


OUT_DIR = "kmeans_nre_results"
os.makedirs(OUT_DIR, exist_ok=True)


# --------------------------
# LOAD & PREPROCESS
# --------------------------
if DATASET_PATH.endswith('.csv'):
    df = pd.read_csv(DATASET_PATH)
elif DATASET_PATH.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(DATASET_PATH)
else:
    raise ValueError("Unsupported file format. Use .csv or .xlsx/.xls")

# detect label column
label_col = None
for col in df.columns:
    if any(k in str(col).lower() for k in ("label", "class", "target")):
        label_col = col
        break

if label_col:
    y_true = df[label_col].values
    df_num = df.drop(columns=[label_col]).select_dtypes(include=[np.number]).copy()
else:
    y_true = None
    df_num = df.select_dtypes(include=[np.number]).copy()

# handle NaN/infs and standardize
df_num = df_num.replace([np.inf, -np.inf], np.nan).fillna(df_num.mean())
X = StandardScaler().fit_transform(df_num.values.astype(np.float64))
n, d = X.shape
print(f"Loaded dataset: {n} samples, {d} numeric features (cleaned)")

# --------------------------
# NREDT-KMeans (MAD OFF)
# Uses Mean + Std threshold instead of Median + MAD
# --------------------------
class NRE_KMeans:

    def __init__(self, n_clusters=3, max_iter=100,
                 tol=1e-4, noise_threshold=2.0,
                 random_state=MASTER_SEED):

        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.noise_threshold = noise_threshold
        self.random_state = random_state

        self.centroids = None
        self.labels = None

    def fit(self, X):

        np.random.seed(self.random_state)

        n_samples = X.shape[0]

        self.centroids = X[
            np.random.choice(
                n_samples,
                self.n_clusters,
                replace=False
            )
        ]

        self.labels = np.full(n_samples, -1)

        for _ in range(self.max_iter):

            prev_centroids = self.centroids.copy()

            distances = np.linalg.norm(
                X[:, np.newaxis] - self.centroids,
                axis=2
            )

            nearest = np.argmin(distances, axis=1)
            nearest_dist = np.min(distances, axis=1)

            # -------------------------------------------------
            # MAD OFF
            # Mean + Standard Deviation Threshold
            # -------------------------------------------------
            mean_dist = np.mean(nearest_dist)
            std_dist = np.std(nearest_dist)

            threshold = mean_dist + self.noise_threshold * std_dist

            # Noise filtering remains ON
            self.labels = np.where(
                nearest_dist > threshold,
                -1,
                nearest
            )

            # Noise-aware centroid update remains ON
            for k in range(self.n_clusters):

                cluster_points = X[self.labels == k]

                if len(cluster_points) > 0:
                    self.centroids[k] = cluster_points.mean(axis=0)

            if np.all(
                np.linalg.norm(
                    self.centroids - prev_centroids,
                    axis=1
                ) < self.tol
            ):
                break

        return self

    def predict(self, X):

        distances = np.linalg.norm(
            X[:, np.newaxis] - self.centroids,
            axis=2
        )

        nearest = np.argmin(distances, axis=1)
        nearest_dist = np.min(distances, axis=1)

        # Mean + Std threshold
        mean_dist = np.mean(nearest_dist)
        std_dist = np.std(nearest_dist)

        threshold = mean_dist + self.noise_threshold * std_dist

        return np.where(
            nearest_dist > threshold,
            -1,
            nearest
        )

    def evaluate(self, X):

        mask = self.labels != -1

        if np.sum(mask) < 2:
            return None

        sil = silhouette_score(
            X[mask],
            self.labels[mask]
        )

        db = davies_bouldin_score(
            X[mask],
            self.labels[mask]
        )

        ch = calinski_harabasz_score(
            X[mask],
            self.labels[mask]
        )

        intra_list = [
            np.mean(
                np.linalg.norm(
                    X[mask][self.labels[mask] == k] -
                    self.centroids[k],
                    axis=1
                )
            )
            for k in range(self.n_clusters)
            if np.any(self.labels[mask] == k)
        ]

        intra = np.mean(intra_list)

        compactness = intra

        inter = np.mean(
            pairwise_distances(self.centroids)
        )

        min_centroid_dist = np.min(
            pairwise_distances(self.centroids)
            + np.eye(self.n_clusters) * 1e12
        )

        xb = (
            np.sum([
                np.sum(
                    np.linalg.norm(
                        X[mask][self.labels[mask] == k]
                        - self.centroids[k],
                        axis=1
                    ) ** 2
                )
                for k in range(self.n_clusters)
                if np.any(self.labels[mask] == k)
            ])
            /
            (X[mask].shape[0] * min_centroid_dist + 1e-12)
        )

        separation_ratio = inter / (intra + 1e-12)

        return {
            "Silhouette": sil,
            "Davies-Bouldin": db,
            "Calinski-Harabasz": ch,
            "Intra": intra,
            "Compactness": compactness,
            "Xie-Beni": xb,
            "Separation-Ratio": separation_ratio,
            "Inter": inter
        }


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

def find_best_k_silhouette(X, k_min=2, k_max=10, seed=42):
    best_k = k_min
    best_score = -1

    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=5, random_state=seed)
        labels = km.fit_predict(X)

        if len(np.unique(labels)) > 1:
            score = silhouette_score(X, labels)
            if score > best_score:
                best_score = score
                best_k = k

    return best_k


# --------------------------
# RUN BASELINE KMEANS
# --------------------------
k_clusters = find_best_k_silhouette(X, 2, 10)
print("Optimal number of clusters:", k_clusters)
print("\nRunning deterministic KMeans...")
kmeans_model = KMeans(n_clusters=k_clusters, n_init=1, max_iter=300, random_state=MASTER_SEED)
kmeans_model.fit(X)
k_labels = kmeans_model.labels_
k_centroids = kmeans_model.cluster_centers_

# compute KMeans metrics (no noise removal)
mask = np.ones(len(X), dtype=bool)

# ------------------
# Intra-cluster distance
# ------------------
intra_list = [
    np.mean(np.linalg.norm(X[k_labels == k] - k_centroids[k], axis=1))
    for k in range(k_clusters)
    if np.any(k_labels == k)
]
intra = np.mean(intra_list)

# ------------------
# Compactness (same as intra but explicitly named)
# ------------------
compactness = intra

# ------------------
# Inter-cluster distance
# ------------------
inter = np.mean(pairwise_distances(k_centroids))

# ------------------
# Xie–Beni Index
# ------------------
min_centroid_dist = np.min(
    pairwise_distances(k_centroids) + np.eye(k_clusters) * 1e12
)

xie_beni = (
    np.sum([
        np.sum(np.linalg.norm(X[k_labels == k] - k_centroids[k], axis=1) ** 2)
        for k in range(k_clusters)
        if np.any(k_labels == k)
    ]) / (X.shape[0] * min_centroid_dist + 1e-12)
)

# ------------------
# Final metric dictionary (ORDER MATCHED)
# ------------------
baseline_scores = {
    "Silhouette": silhouette_score(X[mask], k_labels[mask]),
    "Davies-Bouldin": davies_bouldin_score(X[mask], k_labels[mask]),
    "Calinski-Harabasz": calinski_harabasz_score(X[mask], k_labels[mask]),
    "Intra": intra,
    "Compactness": compactness,
    "Xie-Beni": xie_beni,
    "Separation-Ratio": inter / (intra + 1e-12),
    "Inter": inter
}

# --------------------------
# RUN BASELINE NRE-KMEANS (NO MAD / NO NOISE FILTERING)
# --------------------------
print("\nRunning Baseline NRE-KMeans...")

nre_model = NRE_KMeans(
    n_clusters=k_clusters,
    max_iter=200,
    random_state=MASTER_SEED
)

nre_model.fit(X)

nre_labels = nre_model.labels
nre_centroids = nre_model.centroids

nre_scores = nre_model.evaluate(X)

# ==========================================================
# PART 6 (PART-1)
# FINAL MODEL COMPARISON (30-RUN EXPERIMENT)
# ==========================================================

import pandas as pd
import numpy as np
import os

N_RUNS = 30

print("="*80)
print(f"Running {N_RUNS} independent experiments for all clustering models...")
print("="*80)

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Intra",
    "Compactness",
    "Xie-Beni",
    "Separation-Ratio"
]

# ----------------------------------------------------------
# Store results from all runs
# ----------------------------------------------------------

all_results = {
    "KMeans": {m: [] for m in metrics_names},
    "KMeans++": {m: [] for m in metrics_names},
    "Trimmed-KMeans": {m: [] for m in metrics_names},
    "Robust-KMeans": {m: [] for m in metrics_names},
    "NREDT-KMeans": {m: [] for m in metrics_names},
}

# ----------------------------------------------------------
# Start Multiple Runs
# ----------------------------------------------------------

for run in range(N_RUNS):

    print(f"\n================ RUN {run+1}/{N_RUNS} ================\n")

    CURRENT_SEED = MASTER_SEED + run
# --------------------------
# PART 6 — FINAL TABLE (MEAN ± SD)
# --------------------------
import pandas as pd
import numpy as np
import os

print("\nCalculating full model comparison metrics table...")

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Xie-Beni",
]

all_models_scores = {
    "NREDT-KMeans": nre_scores
}

table_rows = []

for metric in metrics_names:

    row = {"Metric": metric}

    mean_values = {}

    for model_name, scores in all_models_scores.items():

        values = np.asarray(scores[metric], dtype=float)

        mean = np.mean(values)
        sd = np.std(values, ddof=1)

        row[model_name] = f"{mean:.4f} ± {sd:.4f}"
        mean_values[model_name] = mean

    # Decide winner using MEAN only
    if metric in ["Silhouette", "Calinski-Harabasz", "Separation-Ratio"]:
        best_model = max(mean_values, key=mean_values.get)
    else:
        best_model = min(mean_values, key=mean_values.get)

    row["Best_Model"] = best_model

    table_rows.append(row)

# --------------------------
# Final DataFrame
# --------------------------
final_table = pd.DataFrame(table_rows)

print("\n================ FINAL PARAMETER-WISE PERFORMANCE TABLE ================\n")
print(final_table.to_string(index=False))

# --------------------------
# Win Count
# --------------------------
win_counts = final_table["Best_Model"].value_counts()

print("\n" + "="*75)
print("FINAL VERDICT — Model Win Counts")
for model in all_models_scores.keys():
    print(f"{model}: {win_counts.get(model,0)} wins")
print("="*75)

# --------------------------
# Save CSV
# --------------------------
OUT_DIR = "nredt_kmeans_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(OUT_DIR, "All_Model_Comparison_Metrics.csv")
final_table.to_csv(csv_path, index=False)

print(f"\n✔ CSV saved → {csv_path}")

# --------------------------
# Overall Accuracy
# --------------------------
total_metrics = len(metrics_names)

print("\n================ KEY METRIC CLUSTERING QUALITY DECISION ================")
for model in all_models_scores.keys():
    accuracy = win_counts.get(model,0) / total_metrics * 100
    print(f"{model} → {accuracy:.2f}%")
print("=======================================================================\n")

print(DATASET_PATH)
print("=======================================================================\n")

N_RUNS = 5

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Xie-Beni"
]

all_results = {
    "NREDT-KMeans": {m: [] for m in metrics_names}
}

for run in range(N_RUNS):

    print(f"Run {run+1}/{N_RUNS}")

    CURRENT_SEED = MASTER_SEED + run

    # Run your model here
    nre = NRE_KMeans(
        n_clusters=k_clusters,
        random_state=CURRENT_SEED
    )

    nre.fit(X)
    nre_scores = nre.evaluate(X)

    for metric in metrics_names:
        all_results["NREDT-KMeans"][metric].append(float(nre_scores[metric]))

table_rows = []

for metric in metrics_names:

    values = np.array(all_results["NREDT-KMeans"][metric])

    mean = np.mean(values)
    sd = np.std(values, ddof=1)

    table_rows.append({
        "Metric": metric,
        "NREDT-KMeans": f"{mean:.4f} ± {sd:.4f}",
        "Best_Model": "NREDT-KMeans"
    })

final_table = pd.DataFrame(table_rows)

print(final_table.to_string(index=False))


Loaded dataset: 300 samples, 2 numeric features (cleaned)
Optimal number of clusters: 3

Running deterministic KMeans...

Running Baseline NRE-KMeans...
Running 30 independent experiments for all clustering models...

================ RUN 1/30 ================


================ RUN 2/30 ================


================ RUN 3/30 ================


================ RUN 4/30 ================


================ RUN 5/30 ================


================ RUN 6/30 ================


================ RUN 7/30 ================


================ RUN 8/30 ================


================ RUN 9/30 ================


================ RUN 10/30 ================


================ RUN 11/30 ================


================ RUN 12/30 ================


================ RUN 13/30 ================


================ RUN 14/30 ================


================ RUN 15/30 ================


================ RUN 16/30 ================


================ RUN 17/30 ================


=========

NOISE OFF

In [37]:
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    pairwise_distances
)
from sklearn.cluster import KMeans
import openpyxl

# --------------------------
# ENV / DETERMINISM SETTINGS
# --------------------------
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

warnings.filterwarnings("ignore")
MASTER_SEED = 42
random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
GLOBAL_RNG = np.random.default_rng(MASTER_SEED)

# --------------------------
# USER CONFIG - EDIT THIS
# --------------------------
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/circles.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/fashion-mnist_test.csv"  # <-- replace with your dataset path  # <-- replace with your dataset path
#
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/file_1.csv"  # <-- replace with your dataset path
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/cluster_data.csv"  # <-- replace with your dataset path
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/Dry_Bean_Dataset.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/heart_failure_clinical_records_dataset.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/AirfoilSelfNoise.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/Rice_data_type.csv"  # <-- replace with your dataset path

#-----------------------------------------------------------------------------------------------------------------------------------------


OUT_DIR = "kmeans_nre_results"
os.makedirs(OUT_DIR, exist_ok=True)


# --------------------------
# LOAD & PREPROCESS
# --------------------------
if DATASET_PATH.endswith('.csv'):
    df = pd.read_csv(DATASET_PATH)
elif DATASET_PATH.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(DATASET_PATH)
else:
    raise ValueError("Unsupported file format. Use .csv or .xlsx/.xls")

# detect label column
label_col = None
for col in df.columns:
    if any(k in str(col).lower() for k in ("label", "class", "target")):
        label_col = col
        break

if label_col:
    y_true = df[label_col].values
    df_num = df.drop(columns=[label_col]).select_dtypes(include=[np.number]).copy()
else:
    y_true = None
    df_num = df.select_dtypes(include=[np.number]).copy()

# handle NaN/infs and standardize
df_num = df_num.replace([np.inf, -np.inf], np.nan).fillna(df_num.mean())
X = StandardScaler().fit_transform(df_num.values.astype(np.float64))
n, d = X.shape
print(f"Loaded dataset: {n} samples, {d} numeric features (cleaned)")

# --------------------------
# NREDT-KMeans (Noise Filtering OFF)
# MAD Threshold ON
# Standard centroid update using all samples
# --------------------------
class NRE_KMeans:

    def __init__(self, n_clusters=3, max_iter=100,
                 tol=1e-4, noise_threshold=2.0,
                 random_state=MASTER_SEED):

        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.noise_threshold = noise_threshold
        self.random_state = random_state

        self.centroids = None
        self.labels = None

    def fit(self, X):

        np.random.seed(self.random_state)

        n_samples = X.shape[0]

        self.centroids = X[
            np.random.choice(
                n_samples,
                self.n_clusters,
                replace=False
            )
        ]

        for _ in range(self.max_iter):

            prev_centroids = self.centroids.copy()

            distances = np.linalg.norm(
                X[:, np.newaxis] - self.centroids,
                axis=2
            )

            nearest = np.argmin(distances, axis=1)
            nearest_dist = np.min(distances, axis=1)

            # --------------------------
            # MAD threshold (kept)
            # --------------------------
            median_dist = np.median(nearest_dist)
            mad = np.median(
                np.abs(nearest_dist - median_dist)
            ) + 1e-12

            threshold = median_dist + self.noise_threshold * mad

            # --------------------------
            # Noise filtering OFF
            # --------------------------
            self.labels = nearest

            # --------------------------
            # Standard centroid update
            # --------------------------
            for k in range(self.n_clusters):

                cluster_points = X[self.labels == k]

                if len(cluster_points) > 0:
                    self.centroids[k] = cluster_points.mean(axis=0)

            if np.all(
                np.linalg.norm(
                    self.centroids - prev_centroids,
                    axis=1
                ) < self.tol
            ):
                break

        return self

    def predict(self, X):

        distances = np.linalg.norm(
            X[:, np.newaxis] - self.centroids,
            axis=2
        )

        return np.argmin(distances, axis=1)

    def evaluate(self, X):

        labels = self.labels

        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)
        ch = calinski_harabasz_score(X, labels)

        intra = np.mean([
            np.mean(
                np.linalg.norm(
                    X[labels == k] - self.centroids[k],
                    axis=1
                )
            )
            for k in range(self.n_clusters)
            if np.any(labels == k)
        ])

        compactness = intra

        inter = np.mean(
            pairwise_distances(self.centroids)
        )

        min_centroid_dist = np.min(
            pairwise_distances(self.centroids)
            + np.eye(self.n_clusters) * 1e12
        )

        xb = (
            np.sum([
                np.sum(
                    np.linalg.norm(
                        X[labels == k] - self.centroids[k],
                        axis=1
                    ) ** 2
                )
                for k in range(self.n_clusters)
                if np.any(labels == k)
            ])
            /
            (len(X) * min_centroid_dist + 1e-12)
        )

        separation_ratio = inter / (intra + 1e-12)

        return {
            "Silhouette": sil,
            "Davies-Bouldin": db,
            "Calinski-Harabasz": ch,
            "Intra": intra,
            "Compactness": compactness,
            "Xie-Beni": xb,
            "Separation-Ratio": separation_ratio,
            "Inter": inter
        }

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

def find_best_k_silhouette(X, k_min=2, k_max=10, seed=42):
    best_k = k_min
    best_score = -1

    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=5, random_state=seed)
        labels = km.fit_predict(X)

        if len(np.unique(labels)) > 1:
            score = silhouette_score(X, labels)
            if score > best_score:
                best_score = score
                best_k = k

    return best_k


# --------------------------
# RUN BASELINE KMEANS
# --------------------------
k_clusters = find_best_k_silhouette(X, 2, 10)
print("Optimal number of clusters:", k_clusters)
print("\nRunning deterministic KMeans...")
kmeans_model = KMeans(n_clusters=k_clusters, n_init=1, max_iter=300, random_state=MASTER_SEED)
kmeans_model.fit(X)
k_labels = kmeans_model.labels_
k_centroids = kmeans_model.cluster_centers_

# compute KMeans metrics (no noise removal)
mask = np.ones(len(X), dtype=bool)

# ------------------
# Intra-cluster distance
# ------------------
intra_list = [
    np.mean(np.linalg.norm(X[k_labels == k] - k_centroids[k], axis=1))
    for k in range(k_clusters)
    if np.any(k_labels == k)
]
intra = np.mean(intra_list)

# ------------------
# Compactness (same as intra but explicitly named)
# ------------------
compactness = intra

# ------------------
# Inter-cluster distance
# ------------------
inter = np.mean(pairwise_distances(k_centroids))

# ------------------
# Xie–Beni Index
# ------------------
min_centroid_dist = np.min(
    pairwise_distances(k_centroids) + np.eye(k_clusters) * 1e12
)

xie_beni = (
    np.sum([
        np.sum(np.linalg.norm(X[k_labels == k] - k_centroids[k], axis=1) ** 2)
        for k in range(k_clusters)
        if np.any(k_labels == k)
    ]) / (X.shape[0] * min_centroid_dist + 1e-12)
)

# ------------------
# Final metric dictionary (ORDER MATCHED)
# ------------------
baseline_scores = {
    "Silhouette": silhouette_score(X[mask], k_labels[mask]),
    "Davies-Bouldin": davies_bouldin_score(X[mask], k_labels[mask]),
    "Calinski-Harabasz": calinski_harabasz_score(X[mask], k_labels[mask]),
    "Intra": intra,
    "Compactness": compactness,
    "Xie-Beni": xie_beni,
    "Separation-Ratio": inter / (intra + 1e-12),
    "Inter": inter
}

# --------------------------
# RUN BASELINE NRE-KMEANS (NO MAD / NO NOISE FILTERING)
# --------------------------
print("\nRunning Baseline NRE-KMeans...")

nre_model = NRE_KMeans(
    n_clusters=k_clusters,
    max_iter=200,
    random_state=MASTER_SEED
)

nre_model.fit(X)

nre_labels = nre_model.labels
nre_centroids = nre_model.centroids

nre_scores = nre_model.evaluate(X)

# ==========================================================
# PART 6 (PART-1)
# FINAL MODEL COMPARISON (30-RUN EXPERIMENT)
# ==========================================================

import pandas as pd
import numpy as np
import os

N_RUNS = 30

print("="*80)
print(f"Running {N_RUNS} independent experiments for all clustering models...")
print("="*80)

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Intra",
    "Compactness",
    "Xie-Beni",
    "Separation-Ratio"
]

# ----------------------------------------------------------
# Store results from all runs
# ----------------------------------------------------------

all_results = {
    "KMeans": {m: [] for m in metrics_names},
    "KMeans++": {m: [] for m in metrics_names},
    "Trimmed-KMeans": {m: [] for m in metrics_names},
    "Robust-KMeans": {m: [] for m in metrics_names},
    "NREDT-KMeans": {m: [] for m in metrics_names},
}

# ----------------------------------------------------------
# Start Multiple Runs
# ----------------------------------------------------------

for run in range(N_RUNS):

    print(f"\n================ RUN {run+1}/{N_RUNS} ================\n")

    CURRENT_SEED = MASTER_SEED + run
# --------------------------
# PART 6 — FINAL TABLE (MEAN ± SD)
# --------------------------
import pandas as pd
import numpy as np
import os

print("\nCalculating full model comparison metrics table...")

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Xie-Beni",
]

all_models_scores = {
    "NREDT-KMeans": nre_scores
}

table_rows = []

for metric in metrics_names:

    row = {"Metric": metric}

    mean_values = {}

    for model_name, scores in all_models_scores.items():

        values = np.asarray(scores[metric], dtype=float)

        mean = np.mean(values)
        sd = np.std(values, ddof=1)

        row[model_name] = f"{mean:.4f} ± {sd:.4f}"
        mean_values[model_name] = mean

    # Decide winner using MEAN only
    if metric in ["Silhouette", "Calinski-Harabasz", "Separation-Ratio"]:
        best_model = max(mean_values, key=mean_values.get)
    else:
        best_model = min(mean_values, key=mean_values.get)

    row["Best_Model"] = best_model

    table_rows.append(row)

# --------------------------
# Final DataFrame
# --------------------------
final_table = pd.DataFrame(table_rows)

print("\n================ FINAL PARAMETER-WISE PERFORMANCE TABLE ================\n")
print(final_table.to_string(index=False))

# --------------------------
# Win Count
# --------------------------
win_counts = final_table["Best_Model"].value_counts()

print("\n" + "="*75)
print("FINAL VERDICT — Model Win Counts")
for model in all_models_scores.keys():
    print(f"{model}: {win_counts.get(model,0)} wins")
print("="*75)

# --------------------------
# Save CSV
# --------------------------
OUT_DIR = "nredt_kmeans_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(OUT_DIR, "All_Model_Comparison_Metrics.csv")
final_table.to_csv(csv_path, index=False)

print(f"\n✔ CSV saved → {csv_path}")

# --------------------------
# Overall Accuracy
# --------------------------
total_metrics = len(metrics_names)

print("\n================ KEY METRIC CLUSTERING QUALITY DECISION ================")
for model in all_models_scores.keys():
    accuracy = win_counts.get(model,0) / total_metrics * 100
    print(f"{model} → {accuracy:.2f}%")
print("=======================================================================\n")

print(DATASET_PATH)
print("=======================================================================\n")

N_RUNS = 5

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Xie-Beni"
]

all_results = {
    "NREDT-KMeans": {m: [] for m in metrics_names}
}

for run in range(N_RUNS):

    print(f"Run {run+1}/{N_RUNS}")

    CURRENT_SEED = MASTER_SEED + run

    # Run your model here
    nre = NRE_KMeans(
        n_clusters=k_clusters,
        random_state=CURRENT_SEED
    )

    nre.fit(X)
    nre_scores = nre.evaluate(X)

    for metric in metrics_names:
        all_results["NREDT-KMeans"][metric].append(float(nre_scores[metric]))

table_rows = []

for metric in metrics_names:

    values = np.array(all_results["NREDT-KMeans"][metric])

    mean = np.mean(values)
    sd = np.std(values, ddof=1)

    table_rows.append({
        "Metric": metric,
        "NREDT-KMeans": f"{mean:.4f} ± {sd:.4f}",
        "Best_Model": "NREDT-KMeans"
    })

final_table = pd.DataFrame(table_rows)

print(final_table.to_string(index=False))


Loaded dataset: 300 samples, 2 numeric features (cleaned)
Optimal number of clusters: 3

Running deterministic KMeans...

Running Baseline NRE-KMeans...
Running 30 independent experiments for all clustering models...

================ RUN 1/30 ================


================ RUN 2/30 ================


================ RUN 3/30 ================


================ RUN 4/30 ================


================ RUN 5/30 ================


================ RUN 6/30 ================


================ RUN 7/30 ================


================ RUN 8/30 ================


================ RUN 9/30 ================


================ RUN 10/30 ================


================ RUN 11/30 ================


================ RUN 12/30 ================


================ RUN 13/30 ================


================ RUN 14/30 ================


================ RUN 15/30 ================


================ RUN 16/30 ================


================ RUN 17/30 ================


=========

SENSITIVITY CENTROID OFF

In [38]:
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    pairwise_distances
)
from sklearn.cluster import KMeans
import openpyxl

# --------------------------
# ENV / DETERMINISM SETTINGS
# --------------------------
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

warnings.filterwarnings("ignore")
MASTER_SEED = 42
random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
GLOBAL_RNG = np.random.default_rng(MASTER_SEED)

# --------------------------
# USER CONFIG - EDIT THIS
# --------------------------
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/circles.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/fashion-mnist_test.csv"  # <-- replace with your dataset path  # <-- replace with your dataset path
#
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/file_1.csv"  # <-- replace with your dataset path
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/cluster_data.csv"  # <-- replace with your dataset path
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/Dry_Bean_Dataset.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/heart_failure_clinical_records_dataset.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/AirfoilSelfNoise.csv"
#DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset1/Rice_data_type.csv"  # <-- replace with your dataset path

#-----------------------------------------------------------------------------------------------------------------------------------------


OUT_DIR = "kmeans_nre_results"
os.makedirs(OUT_DIR, exist_ok=True)


# --------------------------
# LOAD & PREPROCESS
# --------------------------
if DATASET_PATH.endswith('.csv'):
    df = pd.read_csv(DATASET_PATH)
elif DATASET_PATH.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(DATASET_PATH)
else:
    raise ValueError("Unsupported file format. Use .csv or .xlsx/.xls")

# detect label column
label_col = None
for col in df.columns:
    if any(k in str(col).lower() for k in ("label", "class", "target")):
        label_col = col
        break

if label_col:
    y_true = df[label_col].values
    df_num = df.drop(columns=[label_col]).select_dtypes(include=[np.number]).copy()
else:
    y_true = None
    df_num = df.select_dtypes(include=[np.number]).copy()

# handle NaN/infs and standardize
df_num = df_num.replace([np.inf, -np.inf], np.nan).fillna(df_num.mean())
X = StandardScaler().fit_transform(df_num.values.astype(np.float64))
n, d = X.shape
print(f"Loaded dataset: {n} samples, {d} numeric features (cleaned)")

# ==========================================================
# ABLATION 3 : NREDT − Sensitivity Centroid Update
# (Centroid Refinement OFF)
# ==========================================================

class NRE_KMeans:

    def __init__(self,
                 n_clusters=3,
                 max_iter=100,
                 tol=1e-4,
                 noise_threshold=2.0,
                 random_state=MASTER_SEED):

        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.noise_threshold = noise_threshold
        self.random_state = random_state

        self.centroids = None
        self.labels = None

    def fit(self, X):

        np.random.seed(self.random_state)

        n_samples = X.shape[0]

        self.centroids = X[
            np.random.choice(
                n_samples,
                self.n_clusters,
                replace=False
            )
        ]

        self.labels = np.full(n_samples, -1)

        for _ in range(self.max_iter):

            prev_centroids = self.centroids.copy()

            distances = np.linalg.norm(
                X[:, np.newaxis] - self.centroids,
                axis=2
            )

            nearest = np.argmin(distances, axis=1)
            nearest_dist = np.min(distances, axis=1)

            # --------------------------
            # MAD Noise Detection (ON)
            # --------------------------
            median_dist = np.median(nearest_dist)
            mad = np.median(
                np.abs(nearest_dist - median_dist)
            ) + 1e-12

            threshold = (
                median_dist +
                self.noise_threshold * mad
            )

            self.labels = np.where(
                nearest_dist > threshold,
                -1,
                nearest
            )

            # ------------------------------------------------
            # Centroid Refinement OFF
            # Update centroid using ALL assigned samples
            # (including detected noisy samples)
            # ------------------------------------------------
            for k in range(self.n_clusters):

                cluster_points = X[nearest == k]

                if len(cluster_points) > 0:
                    self.centroids[k] = np.mean(
                        cluster_points,
                        axis=0
                    )

            if np.all(
                np.linalg.norm(
                    self.centroids - prev_centroids,
                    axis=1
                ) < self.tol
            ):
                break

        return self

    def predict(self, X):

        distances = np.linalg.norm(
            X[:, np.newaxis] - self.centroids,
            axis=2
        )

        nearest = np.argmin(distances, axis=1)
        nearest_dist = np.min(distances, axis=1)

        median_dist = np.median(nearest_dist)

        mad = np.median(
            np.abs(nearest_dist - median_dist)
        ) + 1e-12

        threshold = (
            median_dist +
            self.noise_threshold * mad
        )

        return np.where(
            nearest_dist > threshold,
            -1,
            nearest
        )

    def evaluate(self, X):

        mask = self.labels != -1

        if np.sum(mask) < 2:
            return None

        sil = silhouette_score(
            X[mask],
            self.labels[mask]
        )

        db = davies_bouldin_score(
            X[mask],
            self.labels[mask]
        )

        ch = calinski_harabasz_score(
            X[mask],
            self.labels[mask]
        )

        intra = np.mean([
            np.mean(
                np.linalg.norm(
                    X[mask][self.labels[mask] == k]
                    - self.centroids[k],
                    axis=1
                )
            )
            for k in range(self.n_clusters)
            if np.any(self.labels[mask] == k)
        ])

        compactness = intra

        inter = np.mean(
            pairwise_distances(self.centroids)
        )

        min_centroid_dist = np.min(
            pairwise_distances(self.centroids)
            + np.eye(self.n_clusters) * 1e12
        )

        xb = (
            np.sum([
                np.sum(
                    np.linalg.norm(
                        X[mask][self.labels[mask] == k]
                        - self.centroids[k],
                        axis=1
                    ) ** 2
                )
                for k in range(self.n_clusters)
                if np.any(self.labels[mask] == k)
            ])
            /
            (X[mask].shape[0] * min_centroid_dist + 1e-12)
        )

        separation_ratio = inter / (intra + 1e-12)

        return {
            "Silhouette": sil,
            "Davies-Bouldin": db,
            "Calinski-Harabasz": ch,
            "Intra": intra,
            "Compactness": compactness,
            "Xie-Beni": xb,
            "Separation-Ratio": separation_ratio,
            "Inter": inter
        }
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

def find_best_k_silhouette(X, k_min=2, k_max=10, seed=42):
    best_k = k_min
    best_score = -1

    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=5, random_state=seed)
        labels = km.fit_predict(X)

        if len(np.unique(labels)) > 1:
            score = silhouette_score(X, labels)
            if score > best_score:
                best_score = score
                best_k = k

    return best_k


# --------------------------
# RUN BASELINE KMEANS
# --------------------------
k_clusters = find_best_k_silhouette(X, 2, 10)
print("Optimal number of clusters:", k_clusters)
print("\nRunning deterministic KMeans...")
kmeans_model = KMeans(n_clusters=k_clusters, n_init=1, max_iter=300, random_state=MASTER_SEED)
kmeans_model.fit(X)
k_labels = kmeans_model.labels_
k_centroids = kmeans_model.cluster_centers_

# compute KMeans metrics (no noise removal)
mask = np.ones(len(X), dtype=bool)

# ------------------
# Intra-cluster distance
# ------------------
intra_list = [
    np.mean(np.linalg.norm(X[k_labels == k] - k_centroids[k], axis=1))
    for k in range(k_clusters)
    if np.any(k_labels == k)
]
intra = np.mean(intra_list)

# ------------------
# Compactness (same as intra but explicitly named)
# ------------------
compactness = intra

# ------------------
# Inter-cluster distance
# ------------------
inter = np.mean(pairwise_distances(k_centroids))

# ------------------
# Xie–Beni Index
# ------------------
min_centroid_dist = np.min(
    pairwise_distances(k_centroids) + np.eye(k_clusters) * 1e12
)

xie_beni = (
    np.sum([
        np.sum(np.linalg.norm(X[k_labels == k] - k_centroids[k], axis=1) ** 2)
        for k in range(k_clusters)
        if np.any(k_labels == k)
    ]) / (X.shape[0] * min_centroid_dist + 1e-12)
)

# ------------------
# Final metric dictionary (ORDER MATCHED)
# ------------------
baseline_scores = {
    "Silhouette": silhouette_score(X[mask], k_labels[mask]),
    "Davies-Bouldin": davies_bouldin_score(X[mask], k_labels[mask]),
    "Calinski-Harabasz": calinski_harabasz_score(X[mask], k_labels[mask]),
    "Intra": intra,
    "Compactness": compactness,
    "Xie-Beni": xie_beni,
    "Separation-Ratio": inter / (intra + 1e-12),
    "Inter": inter
}

# --------------------------
# RUN BASELINE NRE-KMEANS (NO MAD / NO NOISE FILTERING)
# --------------------------
print("\nRunning Baseline NRE-KMeans...")

nre_model = NRE_KMeans(
    n_clusters=k_clusters,
    max_iter=200,
    random_state=MASTER_SEED
)

nre_model.fit(X)

nre_labels = nre_model.labels
nre_centroids = nre_model.centroids

nre_scores = nre_model.evaluate(X)

# ==========================================================
# PART 6 (PART-1)
# FINAL MODEL COMPARISON (30-RUN EXPERIMENT)
# ==========================================================

import pandas as pd
import numpy as np
import os

N_RUNS = 30

print("="*80)
print(f"Running {N_RUNS} independent experiments for all clustering models...")
print("="*80)

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Intra",
    "Compactness",
    "Xie-Beni",
    "Separation-Ratio"
]

# ----------------------------------------------------------
# Store results from all runs
# ----------------------------------------------------------

all_results = {
    "KMeans": {m: [] for m in metrics_names},
    "KMeans++": {m: [] for m in metrics_names},
    "Trimmed-KMeans": {m: [] for m in metrics_names},
    "Robust-KMeans": {m: [] for m in metrics_names},
    "NREDT-KMeans": {m: [] for m in metrics_names},
}

# ----------------------------------------------------------
# Start Multiple Runs
# ----------------------------------------------------------

for run in range(N_RUNS):

    print(f"\n================ RUN {run+1}/{N_RUNS} ================\n")

    CURRENT_SEED = MASTER_SEED + run
# --------------------------
# PART 6 — FINAL TABLE (MEAN ± SD)
# --------------------------
import pandas as pd
import numpy as np
import os

print("\nCalculating full model comparison metrics table...")

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Xie-Beni",
]

all_models_scores = {
    "NREDT-KMeans": nre_scores
}

table_rows = []

for metric in metrics_names:

    row = {"Metric": metric}

    mean_values = {}

    for model_name, scores in all_models_scores.items():

        values = np.asarray(scores[metric], dtype=float)

        mean = np.mean(values)
        sd = np.std(values, ddof=1)

        row[model_name] = f"{mean:.4f} ± {sd:.4f}"
        mean_values[model_name] = mean

    # Decide winner using MEAN only
    if metric in ["Silhouette", "Calinski-Harabasz", "Separation-Ratio"]:
        best_model = max(mean_values, key=mean_values.get)
    else:
        best_model = min(mean_values, key=mean_values.get)

    row["Best_Model"] = best_model

    table_rows.append(row)

# --------------------------
# Final DataFrame
# --------------------------
final_table = pd.DataFrame(table_rows)

print("\n================ FINAL PARAMETER-WISE PERFORMANCE TABLE ================\n")
print(final_table.to_string(index=False))

# --------------------------
# Win Count
# --------------------------
win_counts = final_table["Best_Model"].value_counts()

print("\n" + "="*75)
print("FINAL VERDICT — Model Win Counts")
for model in all_models_scores.keys():
    print(f"{model}: {win_counts.get(model,0)} wins")
print("="*75)

# --------------------------
# Save CSV
# --------------------------
OUT_DIR = "nredt_kmeans_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_path = os.path.join(OUT_DIR, "All_Model_Comparison_Metrics.csv")
final_table.to_csv(csv_path, index=False)

print(f"\n✔ CSV saved → {csv_path}")

# --------------------------
# Overall Accuracy
# --------------------------
total_metrics = len(metrics_names)

print("\n================ KEY METRIC CLUSTERING QUALITY DECISION ================")
for model in all_models_scores.keys():
    accuracy = win_counts.get(model,0) / total_metrics * 100
    print(f"{model} → {accuracy:.2f}%")
print("=======================================================================\n")

print(DATASET_PATH)
print("=======================================================================\n")

N_RUNS = 5

metrics_names = [
    "Silhouette",
    "Davies-Bouldin",
    "Calinski-Harabasz",
    "Xie-Beni"
]

all_results = {
    "NREDT-KMeans": {m: [] for m in metrics_names}
}

for run in range(N_RUNS):

    print(f"Run {run+1}/{N_RUNS}")

    CURRENT_SEED = MASTER_SEED + run

    # Run your model here
    nre = NRE_KMeans(
        n_clusters=k_clusters,
        random_state=CURRENT_SEED
    )

    nre.fit(X)
    nre_scores = nre.evaluate(X)

    for metric in metrics_names:
        all_results["NREDT-KMeans"][metric].append(float(nre_scores[metric]))

table_rows = []

for metric in metrics_names:

    values = np.array(all_results["NREDT-KMeans"][metric])

    mean = np.mean(values)
    sd = np.std(values, ddof=1)

    table_rows.append({
        "Metric": metric,
        "NREDT-KMeans": f"{mean:.4f} ± {sd:.4f}",
        "Best_Model": "NREDT-KMeans"
    })

final_table = pd.DataFrame(table_rows)

print(final_table.to_string(index=False))


Loaded dataset: 300 samples, 2 numeric features (cleaned)
Optimal number of clusters: 3

Running deterministic KMeans...

Running Baseline NRE-KMeans...
Running 30 independent experiments for all clustering models...

================ RUN 1/30 ================


================ RUN 2/30 ================


================ RUN 3/30 ================


================ RUN 4/30 ================


================ RUN 5/30 ================


================ RUN 6/30 ================


================ RUN 7/30 ================


================ RUN 8/30 ================


================ RUN 9/30 ================


================ RUN 10/30 ================


================ RUN 11/30 ================


================ RUN 12/30 ================


================ RUN 13/30 ================


================ RUN 14/30 ================


================ RUN 15/30 ================


================ RUN 16/30 ================


================ RUN 17/30 ================


=========